# Zusammengeführtes Modell nach MLX/4 Bit konvertieren

Modell und Aufgabe wählen, dann alle Zellen ausführen. Das Ergebnis liegt unter
`omlx_ready/models/` in Drive und wird auf dem Mac nach `~/.omlx/models/` kopiert.

In [ ]:
# @title Modell und Aufgabe
MODEL_ID = "Qwen/Qwen3-4B" # @param ["Qwen/Qwen3-4B", "Qwen/Qwen3-8B", "microsoft/Phi-4-mini-instruct", "mistralai/Mistral-7B-Instruct-v0.3", "meta-llama/Llama-3.1-8B-Instruct"]
TASK = "phishing_origin" # @param ["phishing_origin", "injection"]
MLX_BACKEND = "cpu" # @param ["cpu", "cuda12", "cuda13"]

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'mlx[{MLX_BACKEND}]==0.32.3', 'mlx-lm==0.31.3',
                'transformers==5.15.1', 'safetensors==0.8.0'], check=True)

In [ ]:
import json, shutil
from datetime import datetime, timezone
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT = Path('/content/drive/MyDrive/llm_phishing_finetune_workflow_v3')
RUN_RELATIVE_PATHS = [
    'Qwen--Qwen3-4B/phishing_origin/run2_trainall_seq1024_b641f8bce155',
    'Qwen--Qwen3-4B/injection/run2_trainall_seq1024_ed25182458b1',
    'Qwen--Qwen3-8B/phishing_origin/run2_trainall_seq1024_e8e2f938c346',
    'Qwen--Qwen3-8B/injection/run2_trainall_seq1024_c95e53553674',
    'microsoft--Phi-4-mini-instruct/phishing_origin/run2_trainall_seq1024_ec6e28568e92',
    'microsoft--Phi-4-mini-instruct/injection/run2_trainall_seq1024_6844caf0fa03',
    'mistralai--Mistral-7B-Instruct-v0.3/phishing_origin/run2_trainall_seq1024_f3f4198653be',
    'mistralai--Mistral-7B-Instruct-v0.3/injection/run2_trainall_seq1024_e61fde90cc14',
    'meta-llama--Llama-3.1-8B-Instruct/phishing_origin/run2_trainall_seq1024_da4398c35e8c',
    'meta-llama--Llama-3.1-8B-Instruct/injection/run2_trainall_seq1024_0cfa9daf5a6c',
]
MODEL_NAMES = {
    'Qwen/Qwen3-4B': 'qwen3-4b', 'Qwen/Qwen3-8B': 'qwen3-8b', 'microsoft/Phi-4-mini-instruct': 'phi4-mini-instruct',
    'mistralai/Mistral-7B-Instruct-v0.3': 'mistral-7b-instruct-v0.3', 'meta-llama/Llama-3.1-8B-Instruct': 'llama31-8b-instruct',
}
CURRENT_RUN = next(r for r in RUN_RELATIVE_PATHS if r.startswith(MODEL_ID.replace('/', '--') + '/' + TASK + '/'))

def mlx_convert(source, output):
    """4-Bit-affine Quantisierung (Gruppengröße 64) eines lokalen HF-Ordners nach MLX."""
    import mlx.core as mx
    from mlx_lm import convert
    config = json.loads((source / 'config.json').read_text())
    if 'rope_theta' not in config and isinstance(config.get('rope_parameters'), dict):
        config['rope_theta'] = config['rope_parameters'].get('rope_theta')
        (source / 'config.json').write_text(json.dumps(config, indent=2))
    mx.set_default_device(mx.cpu if MLX_BACKEND == 'cpu' else mx.gpu)
    shutil.rmtree(output, ignore_errors=True)
    convert(str(source), mlx_path=str(output), quantize=True, q_bits=4, q_group_size=64, q_mode='affine', dtype='float16')
    for pattern in ('*.jinja', 'LICENSE*', 'NOTICE*'):
        for path in source.glob(pattern):
            if path.is_file() and not (output / path.name).exists():
                shutil.copy2(path, output / path.name)

def save_to_drive(output, destination, provenance, identity):
    (output / 'artifact_provenance.json').write_text(json.dumps(provenance, indent=2))
    files = {str(p.relative_to(output)): p.stat().st_size for p in sorted(output.rglob('*')) if p.is_file()}
    (output / 'conversion_manifest.json').write_text(json.dumps({
        'manifest_version': 2, 'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'identity': identity, 'device': MLX_BACKEND, 'files': files}, indent=2))
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(output, destination)
    print('Modell in Drive gespeichert:', destination)

In [ ]:
source = PROJECT / 'runs_v4' / CURRENT_RUN / 'merged_hf'
local_source, local_output = Path('/content/merged_hf'), Path('/content/mlx_model')
shutil.rmtree(local_source, ignore_errors=True)
shutil.copytree(source, local_source)
mlx_convert(local_source, local_output)
provenance = json.loads((source / 'artifact_provenance.json').read_text())
destination = PROJECT / 'omlx_ready/models' / f"{MODEL_NAMES[MODEL_ID]}-{TASK.replace('_', '-')}-4bit"
save_to_drive(local_output, destination, provenance,
              {'run': CURRENT_RUN, 'merge_provenance': provenance, 'bits': 4, 'group_size': 64, 'mode': 'affine', 'dtype': 'float16'})